# Corrected LSTM — Fair Evaluation / 修正版 LSTM——公平评估

The original LSTM (`01_lstm_reimplementation.ipynb`) achieved 99.1% accuracy, but EDA revealed two leakage sources:
- `subject` column: perfectly separates fake from real (100% accuracy alone)
- Reuters byline in text: present in 99.2% of real news, 0% of fake news

This notebook retrains the same LSTM architecture with **clean input only**: `title + text`, with `subject` and `date` removed. The goal is to measure how well the model performs when it cannot exploit source-level shortcuts — and to provide a fair baseline for comparison with DistilBERT.

---

原始 LSTM（`01_lstm_reimplementation.ipynb`）达到了 99.1% 的准确率，但 EDA 发现了两个泄露来源：
- `subject` 列：单独使用就能 100% 区分真假新闻
- 正文中的 Reuters 署名：出现在 99.2% 的真新闻中，假新闻中为 0%

本 notebook 使用完全相同的 LSTM 结构重新训练，但输入改为**干净的 `title + text`**，去掉 `subject` 和 `date`。目标是在模型无法利用来源捷径的情况下，测量其真实的语言理解能力，并为后续与 DistilBERT 的比较提供公平的基准。

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

os.makedirs('../results', exist_ok=True)
print('TensorFlow:', tf.__version__)

/Users/sl1425/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow: 2.16.2


## 2. Load Data — Clean Input Only

与原始 LSTM 的关键区别：输入改为 `title + text` 拼接，去掉 `subject` 和 `date` 列，消除泄露来源。

Key difference from the original LSTM: input is now `title + text` concatenated. `subject` and `date` are excluded to remove leakage sources.

In [ ]:
fake = pd.read_csv('../data/raw/Fake.csv')
true = pd.read_csv('../data/raw/True.csv')

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true]).sample(frac=1, random_state=42).reset_index(drop=True)

# 问题一：去掉开头的 Reuters 署名，支持多词城市名
# Fix 1: Remove Reuters byline at start, handles multi-word city names
# e.g. "NEW YORK (Reuters) -" or "WASHINGTON (Reuters) -"
df['text'] = df['text'].str.replace(r'[\w\s]+\(Reuters\)\s*-?\s*', '', regex=True)

# 问题二：去掉正文中其他地方出现的 Reuters 字样
# Fix 2: Remove remaining Reuters mentions anywhere in the body
df['text'] = df['text'].str.replace(r'Reuters', '', regex=False)

# 问题三：去掉去掉 Reuters 后可能残留的开头全大写城市署名
# Fix 3: Remove remaining all-caps city datelines at start of text
# e.g. "WASHINGTON -" left after Reuters removal
df['text'] = df['text'].str.replace(r'^\s*[A-Z][A-Z\s\.]+\s*-\s*', '', regex=True)

# 拼接 title 和 text，去掉 subject 和 date
# Concatenate title and text — subject and date excluded
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

print(f'Total: {len(df):,} rows')
print('Leakage removed:')
print('  - subject, date columns excluded')
print('  - Reuters byline (multi-word) stripped from start of text')
print('  - All remaining Reuters mentions removed from body')
print('  - Residual all-caps city datelines removed from start of text')
df[['content', 'label']].head(3)

四个来源层面的泄露信号全部消除：`subject` 和 `date` 列排除；Reuters 署名（含多词城市名）清除；正文中其他 Reuters 字样清除；去掉署名后残留的全大写城市署名（如 `WASHINGTON -`）也一并清除。

剩余的潜在偏差是 Reuters 的**写作风格**（正式语气、倒金字塔结构、引用方式），技术上难以消除，将在 error analysis 中作为模型局限性提出。

All four source-level leakage signals are removed: `subject` and `date` excluded; Reuters byline (multi-word) stripped; remaining Reuters mentions removed; residual all-caps city datelines (e.g. `WASHINGTON -`) also cleaned.

The remaining potential bias is Reuters' **writing style** (formal tone, inverted pyramid, citation patterns), which cannot be easily removed and will be noted as a model limitation in error analysis.

## 3. Text Preprocessing / 文本预处理

按照原论文的流程，文本预处理分三步：划分训练测试集 → Tokenizer 建立词表 → 将文本转换为数字序列并统一长度（padding）。输入使用 `content`（title + text），不含 `subject` 和 `date`。

Following the original paper pipeline: train/test split → Tokenizer vocabulary construction → convert text to padded integer sequences. Input is `content` (title + text only).

In [ ]:
# 划分训练测试集 80/20，与原论文一致
# Train/test split 80/20, matching original paper
X = df['content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
# Tokenizer：保留前 5000 高频词，与原论文一致
# Tokenizer: top 5,000 words, matching original paper
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index)
print(f'Full vocabulary size: {vocab_size:,}')
print(f'Using top 5,000 words (as per original paper)')

In [ ]:
# 转为数字序列并补齐至长度 200，与原论文一致
# Convert to sequences and pad to length 200, matching original paper
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=200)
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=200)

print(f'X_train shape: {X_train_seq.shape}')
print(f'X_test shape:  {X_test_seq.shape}')

预处理完成。每篇文章被表示为长度 200 的整数数组，不足补零，超过截断。与原始 LSTM 的唯一区别是输入内容：这里是 `title + text`，原始版本是 `text`。

Preprocessing complete. Each article is a zero-padded integer sequence of length 200. The only difference from the original LSTM is the input: here we use `title + text`; the original used `text` only.